# Text Classification: Modelling & Experiment Tracking

Notebook ini berisi langkah-langkah untuk:
1. **Modelling**: Melatih beberapa algoritma Machine Learning (Logistic Regression, Random Forest, Linear SVC, Naive Bayes).
2. **Log Experiment**: Mencatat hyperparameter, metrik evaluasi (Accuracy, Precision, Recall, F1-Score), dan waktu training dari setiap model yang dilatih.
3. **Model Terbaik**: Membandingkan hasil eksperimen dan mencari model dengan performa terbaik berdasarkan F1-Score.

Data yang digunakan adalah data hasil pembersihan dari tahap sebelumnya.

In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from imblearn.over_sampling import SMOTE
import warnings
import urllib3
urllib3.disable_warnings()
warnings.filterwarnings('ignore')
import mlflow
import mlflow.sklearn

/Users/enricko/Kuliah/Data Mining/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


### 1. Load Dataset & Preprocessing

In [2]:
df = pd.read_csv('results/data/dataset_cleaned.csv')
df = df.dropna(subset=['cleaned_text', 'sentiment'])

print("Total data:", len(df))
print("\nDistribusi Kelas:")
print(df['sentiment'].value_counts())

Total data: 59591

Distribusi Kelas:
sentiment
happiness     11942
sadness       10944
worry         10782
neutral        8433
love           5431
surprise       2892
anger          2819
fun            1774
relief         1521
hate           1318
empty           797
enthusiasm      759
boredom         179
Name: count, dtype: int64


In [3]:
# Ekstraksi Fitur (TF-IDF)
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['cleaned_text'])
y = df['sentiment']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Handle Imbalance dengan SMOTE
print("Menerapkan SMOTE pada data training...")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Distribusi kelas setelah SMOTE:")
print(pd.Series(y_train_smote).value_counts())

Menerapkan SMOTE pada data training...
Distribusi kelas setelah SMOTE:
sentiment
worry         9510
neutral       9510
love          9510
sadness       9510
happiness     9510
surprise      9510
relief        9510
anger         9510
fun           9510
enthusiasm    9510
empty         9510
hate          9510
boredom       9510
Name: count, dtype: int64


### 2. Log Experiment & Modelling

In [4]:
# Setup untuk Log Experiment
import mlflow
import mlflow.sklearn

experiment_logs = []

# Set experiment name
mlflow.set_experiment("Emotion_Classification_Experiment")

def log_experiment(model_name, model, X_train, y_train, X_test, y_test, params="Default"):
    print(f"Training {model_name}...")
    
    with mlflow.start_run(run_name=model_name):
        start_time = time.time()
        
        # Train
        model.fit(X_train, y_train)
        training_time = time.time() - start_time
        
        # Predict
        y_pred = model.predict(X_test)
        
        # Metrics
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        
        # Log to MLflow
        mlflow.log_param("model_type", model_name)
        mlflow.log_param("parameters", params)
        mlflow.log_metric("training_time", training_time)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall", rec)
        mlflow.log_metric("f1_score", f1)
        
        # Log Model
        mlflow.sklearn.log_model(model, name="model", input_example=X_train[:1])
        
        # Simpan ke log manual juga (untuk dataframe)
        experiment_logs.append({
            'Model': model_name,
            'Parameters': params,
            'Training Time (s)': round(training_time, 4),
            'Accuracy': round(acc, 4),
            'Precision': round(prec, 4),
            'Recall': round(rec, 4),
            'F1-Score': round(f1, 4)
        })
        
        print(f"{model_name} selesai. F1-Score: {f1:.4f}\n")
        return model


In [5]:
# Melatih berbagai model Machine Learning

# Model 1: Logistic Regression
lr_model = LogisticRegression(max_iter=1000, multi_class='ovr', random_state=42)
log_experiment("Logistic Regression", lr_model, X_train_smote, y_train_smote, X_test, y_test, params="max_iter=1000, multi_class=ovr")

# Model 2: Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
log_experiment("Random Forest", rf_model, X_train_smote, y_train_smote, X_test, y_test, params="n_estimators=100")

# Model 3: Linear SVC (SVM)
svm_model = LinearSVC(random_state=42)
log_experiment("Linear SVC", svm_model, X_train_smote, y_train_smote, X_test, y_test, params="default")

# Model 4: Naive Bayes
nb_model = MultinomialNB()
log_experiment("Naive Bayes", nb_model, X_train_smote, y_train_smote, X_test, y_test, params="default")

Training Logistic Regression...
Logistic Regression selesai. F1-Score: 0.4729

Training Random Forest...
Random Forest selesai. F1-Score: 0.4707

Training Linear SVC...
Linear SVC selesai. F1-Score: 0.4528

Training Naive Bayes...
Naive Bayes selesai. F1-Score: 0.4309



MultinomialNB()

### 3. Evaluasi Eksperimen & Model Terbaik

In [6]:
# Tampilkan hasil log eksperimen
df_logs = pd.DataFrame(experiment_logs)
df_logs = df_logs.sort_values(by='F1-Score', ascending=False).reset_index(drop=True)

print("=== LOG EXPERIMENT ===")
display(df_logs)

# Simpan log ke CSV
import os
if not os.path.exists('results'):
    os.makedirs('results')
df_logs.to_csv('results/experiment_logs.csv', index=False)
print("\nLog eksperimen disimpan ke 'results/experiment_logs.csv'")

=== LOG EXPERIMENT ===


,Model,Parameters,Training Time (s),Accuracy,Precision,Recall,F1-Score
0,Logistic Regression,"max_iter=1000, multi_class=ovr",2.2164,0.4367,0.5395,0.4367,0.4729
1,Random Forest,n_estimators=100,66.5354,0.4423,0.5190,0.4423,0.4707
2,Linear SVC,default,2.8178,0.4217,0.5113,0.4217,0.4528
3,Naive Bayes,default,0.1812,0.3885,0.5258,0.3885,0.4309



Log eksperimen disimpan ke 'results/experiment_logs.csv'


In [7]:
# 3. Model Terbaik
best_model_row = df_logs.iloc[0]

print("="*50)
print("\U0001F389 MODEL TERBAIK \U0001F389")
print("="*50)
print(f"Model: {best_model_row['Model']}")
print(f"Parameters: {best_model_row['Parameters']}")
print(f"F1-Score: {best_model_row['F1-Score']}")
print(f"Accuracy: {best_model_row['Accuracy']}")
print("="*50)

# Menampilkan classification report dari model terbaik
best_model_name = best_model_row['Model']
best_model_instance = None

if best_model_name == "Logistic Regression":
    best_model_instance = lr_model
elif best_model_name == "Random Forest":
    best_model_instance = rf_model
elif best_model_name == "Linear SVC":
    best_model_instance = svm_model
elif best_model_name == "Naive Bayes":
    best_model_instance = nb_model

if best_model_instance:
    print("\nClassification Report untuk Model Terbaik:")
    y_pred_best = best_model_instance.predict(X_test)
    print(classification_report(y_test, y_pred_best, zero_division=0))

🎉 MODEL TERBAIK 🎉
Model: Logistic Regression
Parameters: max_iter=1000, multi_class=ovr
F1-Score: 0.4729
Accuracy: 0.4367

Classification Report untuk Model Terbaik:
              precision    recall  f1-score   support

       anger       0.75      0.87      0.81       564
     boredom       0.01      0.09      0.02        35
       empty       0.04      0.18      0.07       151
  enthusiasm       0.03      0.12      0.04       143
         fun       0.11      0.21      0.14       378
   happiness       0.74      0.58      0.65      2432
        hate       0.14      0.37      0.21       255
        love       0.53      0.53      0.53      1093
     neutral       0.32      0.22      0.26      1687
      relief       0.10      0.27      0.15       293
     sadness       0.77      0.55      0.64      2234
    surprise       0.29      0.31      0.30       565
       worry       0.52      0.31      0.39      2089

    accuracy                           0.44     11919
   macro avg       0.3

In [8]:
# Menjalankan MLflow UI langsung dari Notebook
print("Memulai MLflow UI di background...")
print("Silakan klik link berikut: http://127.0.0.1:5050")   
get_ipython().system_raw('mlflow ui --port 5050 &')

Memulai MLflow UI di background...
Silakan klik link berikut: http://127.0.0.1:5050


/Users/enricko/Kuliah/Data Mining/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
[2026-04-27 15:54:22 +0700] [37768] [INFO] Starting gunicorn 23.0.0
[2026-04-27 15:54:22 +0700] [37768] [ERROR] Connection in use: ('127.0.0.1', 5050)
[2026-04-27 15:54:22 +0700] [37768] [ERROR] connection to ('127.0.0.1', 5050) failed: [Errno 48] Address already in use
[2026-04-27 15:54:23 +0700] [37768] [ERROR] Connection in use: ('127.0.0.1', 5050)
[2026-04-27 15:54:23 +0700] [37768] [ERROR] connection to ('127.0.0.1', 5050) failed: [Errno 48] Address already in use
[2026-04-27 15:54:24 +0700] [37768] [ERROR] Connection in use: ('127.0.0.1', 5050)
[2026-04-27 15:54:24 +0700] [37768] [ERROR] connection to ('127.0.0.1', 5050) failed: [Errno 48] Address already in use
[2026-04-27 15:54:25 +0700] [37768] [ERRO